## Name: Asit Jain
## Roll No: M25DE1049
## Assignment 3 - MLBD

# Part 2: Collaborative Filtering
## Task 3: User-Based Collaborative Filtering

Recommend items based on preferences of similar users. Users with similar tastes in the past will have similar preferences in the future.

**Dataset:** MovieLens ml-latest-small

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'scikit-learn', 'numpy', 'scipy', '-q'])

import os
import zipfile
import urllib.request
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
print('All imports successful!')

All imports successful!


### Step 1: Load Dataset and Build User-Movie Rating Matrix

In [2]:
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_path = "ml-latest-small.zip"
extract_dir = "ml-latest-small"

if not os.path.exists(extract_dir):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('.')
    print("Downloaded and extracted.")
else:
    print("Dataset already exists.")

movies = pd.read_csv(os.path.join(extract_dir, 'movies.csv'))
ratings = pd.read_csv(os.path.join(extract_dir, 'ratings.csv'))
print(f"Movies: {len(movies)}, Ratings: {len(ratings)}, Users: {ratings['userId'].nunique()}")

Dataset already exists.
Movies: 9742, Ratings: 100836, Users: 610


In [3]:
# Build user-movie rating matrix (users as rows, movies as columns)
rating_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating')
print(f"Rating matrix shape: {rating_matrix.shape}")
print(f"Sparsity: {rating_matrix.isna().sum().sum() / (rating_matrix.shape[0] * rating_matrix.shape[1]) * 100:.2f}%")

Rating matrix shape: (610, 9724)
Sparsity: 98.30%


### Step 2: Compute User-User Similarity (Pearson Correlation)

We use Pearson correlation rather than cosine similarity because it accounts for different rating biases — some users rate generously while others rate harshly. Pearson correlation centers ratings around each user's mean, making it more robust to these biases.

In [4]:
# Compute Pearson correlation between users
# We use the rating matrix transposed so corr() computes user-user correlation
user_similarity = rating_matrix.T.corr(method='pearson')
print(f"User similarity matrix shape: {user_similarity.shape}")
user_similarity.iloc[:5, :5]

User similarity matrix shape: (610, 610)


userId,1,2,3,4,5
userId,,,,,
1,1.000000,NaN,0.079819,0.207983,0.268749
2,NaN,1.0,NaN,NaN,NaN
3,0.079819,NaN,1.000000,NaN,NaN
4,0.207983,NaN,NaN,1.000000,-0.336525
5,0.268749,NaN,NaN,-0.336525,1.000000


### Step 3: Predict Ratings Using Weighted Average of K Nearest Neighbors

In [5]:
def predict_rating(user_id, movie_id, rating_matrix, user_similarity, K=30):
    """
    Predict rating for a user-movie pair using K most similar users
    who have rated that movie.
    
    Uses the weighted average formula:
    pred(u, m) = mean(u) + sum(sim(u,v) * (r(v,m) - mean(v))) / sum(|sim(u,v)|)
    """
    if movie_id not in rating_matrix.columns:
        return np.nan
    
    # Users who rated this movie
    movie_ratings = rating_matrix[movie_id].dropna()
    
    # Exclude the target user
    other_users = movie_ratings.drop(user_id, errors='ignore')
    if other_users.empty:
        return np.nan
    
    # Get similarities with these users
    sims = user_similarity.loc[user_id, other_users.index].dropna()
    if sims.empty:
        return np.nan
    
    # Select top-K most similar users (by absolute similarity)
    top_k = sims.abs().nlargest(K).index
    top_sims = sims[top_k]
    top_ratings = other_users[top_k]
    
    # Filter out zero similarities
    mask = top_sims != 0
    top_sims = top_sims[mask]
    top_ratings = top_ratings[mask]
    
    if top_sims.empty:
        return np.nan
    
    # Weighted average with mean-centering
    user_mean = rating_matrix.loc[user_id].mean()
    neighbor_means = rating_matrix.loc[top_sims.index].mean(axis=1)
    
    numerator = (top_sims * (top_ratings - neighbor_means)).sum()
    denominator = top_sims.abs().sum()
    
    prediction = user_mean + numerator / denominator
    # Clip to valid rating range
    return np.clip(prediction, 0.5, 5.0)

# Quick test
test_pred = predict_rating(1, 1, rating_matrix, user_similarity, K=30)
actual = rating_matrix.loc[1, 1] if 1 in rating_matrix.columns else 'N/A'
print(f"User 1, Movie 1 — Predicted: {test_pred:.2f}, Actual: {actual}")

User 1, Movie 1 — Predicted: 4.49, Actual: 4.0


### Step 4: Recommendation Function

In [6]:
def recommend_user_cf(user_id, rating_matrix, user_similarity, movies_df, K=30, top_n=10):
    """
    Recommend top-N movies for a user based on predicted ratings.
    Only predicts for movies the user has NOT rated.
    """
    # Movies the user hasn't rated
    user_ratings = rating_matrix.loc[user_id]
    unrated = user_ratings[user_ratings.isna()].index.tolist()
    
    # Predict ratings for unrated movies (sample for speed)
    # In practice, we'd predict for all; here we limit to keep it fast
    predictions = {}
    for movie_id in unrated:
        pred = predict_rating(user_id, movie_id, rating_matrix, user_similarity, K=K)
        if not np.isnan(pred):
            predictions[movie_id] = pred
    
    # Sort by predicted rating
    top_movies = sorted(predictions.items(), key=lambda x: x[1], reverse=True)[:top_n]
    
    results = pd.DataFrame({
        'movieId': [m for m, _ in top_movies],
        'Predicted Rating': [round(r, 2) for _, r in top_movies]
    }).merge(movies_df[['movieId', 'title', 'genres']], on='movieId')
    
    return results[['title', 'genres', 'Predicted Rating']]

### Step 5: Test Recommendations

In [7]:
# Show user 1's actual top-rated movies
user1_rated = ratings[ratings['userId'] == 1].merge(movies, on='movieId')
print("User 1's top-rated movies:")
print(user1_rated.sort_values('rating', ascending=False)[['title', 'genres', 'rating']].head(10).to_string(index=False))

print("\n" + "="*70)
print("Top 10 Recommendations for User 1 (User-Based CF, K=30)")
print("="*70)
recommend_user_cf(1, rating_matrix, user_similarity, movies, K=30, top_n=10)

User 1's top-rated movies:
                                 title                             genres  rating
           Seven (a.k.a. Se7en) (1995)                   Mystery|Thriller     5.0
            Usual Suspects, The (1995)             Crime|Mystery|Thriller     5.0
                  Bottle Rocket (1996)     Adventure|Comedy|Crime|Romance     5.0
Dumb & Dumber (Dumb and Dumber) (1994)                   Adventure|Comedy     5.0
                  Billy Madison (1995)                             Comedy     5.0
                      Desperado (1995)             Action|Romance|Western     5.0
                 Canadian Bacon (1995)                         Comedy|War     5.0
                        Rob Roy (1995)           Action|Drama|Romance|War     5.0
                      Pinocchio (1940) Animation|Children|Fantasy|Musical     5.0
                      Tombstone (1993)               Action|Drama|Western     5.0

Top 10 Recommendations for User 1 (User-Based CF, K=30)


,title,genres,Predicted Rating
0,Shanghai Triad (Yao a yao yao dao waipo qiao) ...,Crime|Drama,5.0
1,"Postman, The (Postino, Il) (1994)",Comedy|Drama|Romance,5.0
2,Chungking Express (Chung Hing sam lam) (1994),Drama|Mystery|Romance,5.0
3,Pie in the Sky (1996),Comedy|Romance,5.0
4,"Awfully Big Adventure, An (1995)",Drama,5.0
5,Amateur (1994),Crime|Drama|Thriller,5.0
6,Blue in the Face (1995),Comedy|Drama,5.0
7,Mixed Nuts (1994),Comedy,5.0
8,Nina Takes a Lover (1994),Comedy|Romance,5.0
9,Shallow Grave (1994),Comedy|Drama|Thriller,5.0


In [8]:
print("="*70)
print("Top 10 Recommendations for User 5 (User-Based CF, K=30)")
print("="*70)
recommend_user_cf(5, rating_matrix, user_similarity, movies, K=30, top_n=10)

Top 10 Recommendations for User 5 (User-Based CF, K=30)


,title,genres,Predicted Rating
0,"Addiction, The (1995)",Drama|Horror,5.0
1,Jason's Lyric (1994),Crime|Drama,5.0
2,In the Realm of the Senses (Ai no corrida) (1976),Drama,5.0
3,Denise Calls Up (1995),Comedy,5.0
4,Land Before Time III: The Time of the Great Gi...,Adventure|Animation|Children|Musical,5.0
5,My Man Godfrey (1936),Comedy|Romance,5.0
6,To Gillian on Her 37th Birthday (1996),Drama|Romance,5.0
7,Hype! (1996),Documentary,5.0
8,Vampire in Venice (Nosferatu a Venezia) (Nosfe...,Horror,5.0
9,Leave It to Beaver (1997),Comedy,5.0


### Step 5: Evaluate — RMSE, Precision@K, Recall@K

In [ ]:
def evaluate_user_cf(ratings_df, movies_df, K_neighbors=30, K_rec=10, threshold=4.0, n_test_users=50):
    """
    Evaluate user-based CF using train/test split.
    Computes RMSE on predicted vs actual ratings, and Precision@K / Recall@K.
    """
    # Train/test split
    train_df, test_df = train_test_split(ratings_df, test_size=0.2, random_state=42)
    
    # Build rating matrix from train set
    train_matrix = train_df.pivot_table(index='userId', columns='movieId', values='rating')
    train_sim = train_matrix.T.corr(method='pearson')
    
    # --- RMSE ---
    # Sample test ratings for speed
    test_sample = test_df[test_df['userId'].isin(train_matrix.index)].head(500)
    actuals, preds = [], []
    
    for _, row in test_sample.iterrows():
        uid, mid, actual = int(row['userId']), int(row['movieId']), row['rating']
        if mid not in train_matrix.columns or uid not in train_sim.index:
            continue
        pred = predict_rating(uid, mid, train_matrix, train_sim, K=K_neighbors)
        if not np.isnan(pred):
            actuals.append(actual)
            preds.append(pred)
    
    rmse = np.sqrt(mean_squared_error(actuals, preds)) if actuals else float('nan')
    
    # --- Precision@K and Recall@K ---
    precisions, recalls = [], []
    eval_users = test_df['userId'].unique()[:n_test_users]
    
    for uid in eval_users:
        if uid not in train_matrix.index:
            continue
        
        # Ground truth: highly rated movies in test set
        user_test = test_df[(test_df['userId'] == uid) & (test_df['rating'] >= threshold)]
        relevant = set(user_test['movieId'])
        if not relevant:
            continue
        
        # Get recommendations
        recs = recommend_user_cf(uid, train_matrix, train_sim, movies_df, K=K_neighbors, top_n=K_rec)
        if recs is None or recs.empty:
            continue
        rec_ids = set(recs.merge(movies_df, left_on='title', right_on='title')['movieId'])
        
        hits = rec_ids & relevant
        precisions.append(len(hits) / K_rec)
        recalls.append(len(hits) / len(relevant))
    
    return {
        'RMSE': rmse,
        'Precision@K': np.mean(precisions) if precisions else 0,
        'Recall@K': np.mean(recalls) if recalls else 0,
        'RMSE_samples': len(actuals),
        'P/R_users': len(precisions)
    }

print("Evaluating (this may take a minute)...")
results = evaluate_user_cf(ratings, movies, K_neighbors=30, K_rec=10)
print(f"\nRMSE: {results['RMSE']:.4f} (on {results['RMSE_samples']} predictions)")
print(f"Precision@10: {results['Precision@K']:.4f}")
print(f"Recall@10: {results['Recall@K']:.4f}")
print(f"Evaluated users for P/R: {results['P/R_users']}")

Evaluating (this may take a minute)...


### Step 6: Test with Different Values of K

In [ ]:
print("Testing different K values (number of similar users)...\n")
print(f"{'K':>4s} | {'RMSE':>8s} | {'Prec@10':>8s} | {'Recall@10':>9s}")
print("-" * 40)

for K in [5, 10, 20, 30, 50]:
    res = evaluate_user_cf(ratings, movies, K_neighbors=K, K_rec=10, n_test_users=30)
    print(f"{K:>4d} | {res['RMSE']:>8.4f} | {res['Precision@K']:>8.4f} | {res['Recall@K']:>9.4f}")